# 01 – Chuẩn bị dữ liệu CREMA-D cho pilot (chạy một lần)
**Cài đặt notebook:** Accelerator = *None* (CPU là đủ); **Internet = On** (cần tài khoản đã xác minh số điện thoại).
**Cách chạy:** bấm *Save Version → Save & Run All (Commit)*. Khi chạy xong, output `cremad_processed.tar` (khoảng 1,6 GB) được lưu cùng version này.
Notebook `02_train_pilot` sẽ dùng output này làm input.

In [ ]:
import os, subprocess, sys, importlib
print('CPUs:', os.cpu_count())
subprocess.run('df -h /tmp /kaggle/working', shell=True)
subprocess.run('rm -rf /tmp/icme && git clone -q --depth 1 -b claude/eager-noether-4f8c81 https://github.com/ptrnghieu/icme.git /tmp/icme', shell=True, check=True)
for mod, pkg in [('librosa', 'librosa'), ('cv2', 'opencv-python-headless'), ('soundfile', 'soundfile')]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
os.chdir('/tmp/icme/pilot')
print(subprocess.run('git -C /tmp/icme log -1 --format=%h_%cd', shell=True, capture_output=True, text=True).stdout)

In [ ]:
# Tải khoảng 3 GB (WAV + FLV) qua media.githubusercontent.com và tiền xử lý giống OGM-GE.
# Raw được đặt ở /tmp và bị xóa sau khi xử lý.
!python prepare_cremad.py --root /tmp/cremad --raw_dir /tmp/cremad_raw --workers 16 --delete_raw

In [ ]:
!python human_reliability.py --root /tmp/cremad
!python make_splits.py --root /tmp/cremad

In [ ]:
import pandas as pd
m = pd.read_csv('/tmp/cremad/manifest.csv')
ok = (m.spec_ok == 1) & (m.n_frames > 0)
print('usable clips:', ok.sum(), '/', len(m), '(expected 7442 / 7442)')
print(m.loc[~ok].head(20))

In [ ]:
# Đóng gói thành 1 file để output gọn, tránh giới hạn số lượng file.
!tar -cf /kaggle/working/cremad_processed.tar -C /tmp/cremad spec frames_fps1 splits manifest.csv human_reliability.csv finishedResponses.csv
!ls -la /kaggle/working